# Lecture 4 — Agentic Resume Outreach

**Goal**: Build an agent that scores resumes and drafts personalized outreach emails.

Our agent will:
1. **Score** each candidate using the optimized prompt from Lecture 3
2. **Decide** an outcome (interview, reject, or flag for review) based on the score
3. **Draft** a personalized email appropriate to the outcome
4. **Submit** the result to the leaderboard

## The Agent Loop
```
while not done:
    observe(action_history)
    decide(which_tool_to_call)
    execute(tool)        # ← tools make REAL LLM calls
    log(result)
```

In [ ]:
import json
import random
import pandas as pd
from IPython.display import display, HTML

from agent_utils import (
    load_resumes,
    load_job_requirements,
    structured_llm_call,
    submit_outreach,
    delete_outreach,
    delete_team,
    SCORING_PROMPT_TEMPLATE,
)

random.seed(42)

# Configuration
OPENROUTER_API_KEY = ""  # <-- Paste your OpenRouter API key here
TEAM_NAME = ""  # <-- Your team name here
MODEL = "anthropic/claude-sonnet-4-6"

if not OPENROUTER_API_KEY.strip():
    raise RuntimeError("Please set OPENROUTER_API_KEY above")
if not TEAM_NAME.strip():
    raise RuntimeError("Please set TEAM_NAME above")

# Load data
resumes = load_resumes('../data/resumes_final_with_gold_silver.csv')
job_req = load_job_requirements('../data/job_req_senior.md')

# Build the scoring prompt
scoring_prompt = SCORING_PROMPT_TEMPLATE.format(job_req=job_req)

# Select candidates: 2 gold + 2 silver + 4 wild
gold = {rid: r for rid, r in resumes.items() if rid.startswith('g')}
silver = {rid: r for rid, r in resumes.items() if rid.startswith('s')}
wild = {rid: r for rid, r in resumes.items() if rid[0].isdigit()}
wild_sample = dict(random.sample(list(wild.items()), 4))

candidates = {**gold, **silver, **wild_sample}

print(f"Model: {MODEL}")
print(f"Team: {TEAM_NAME}")
print(f"Candidates: {len(gold)} gold + {len(silver)} silver + {len(wild_sample)} wild = {len(candidates)}")
print(f"IDs: {sorted(candidates.keys())}")

## Step 1: Define Our Tools

Our agent has three tools. Unlike Lecture 4's original mock tools, these make **real LLM calls** — the agent genuinely needs the loop because it can't draft an email until it knows the score.

| Tool | What it does | Why the agent needs it |
|------|-------------|----------------------|
| `score_resume` | Runs the L3 optimized scorer | Agent doesn't know the score until it calls this |
| `draft_outreach_email` | Generates a personalized email | Content depends on the score and outcome |
| `done` | Signals completion | Tells the loop to stop |

In [ ]:
# Tool 1: Score a resume using the Lecture 3 optimized prompt
def score_resume(candidate_id):
    """Score a candidate's resume. Returns score (0-100) and reasoning."""
    resume = resumes[candidate_id]
    result = structured_llm_call(
        api_key=OPENROUTER_API_KEY,
        prompt=scoring_prompt,
        context_data={"resume": resume['Resume_str']},
        output_schema={"score": "integer 0-100", "reasoning": "1-3 sentence explanation"},
        model=MODEL,
        temperature=0.2,
    )
    if result['error']:
        return {"status": "error", "message": result['error']}
    
    score = result['result'].get('score', 0)
    reasoning = result['result'].get('reasoning', '')
    return {
        "status": "success",
        "score": score,
        "reasoning": reasoning,
        "message": f"Score: {score}/100 — {reasoning}",
        "usage": result['usage'],
    }


# Tool 2: Draft a personalized outreach email
def draft_outreach_email(candidate_id, outcome, key_points):
    """Draft a professional email based on the routing outcome."""
    
    email_prompt = f"""Write a professional email from a hiring manager to a job candidate.

OUTCOME: {outcome}
KEY POINTS ABOUT THIS CANDIDATE: {key_points}

GUIDELINES:
- If INTERVIEW: Express enthusiasm about their background. Mention specific strengths from the key points. Invite them to schedule a technical interview. Keep it warm and professional.
- If REJECT: Be respectful and encouraging. Thank them for applying. Briefly note why they weren't selected (without being harsh). Wish them well.  
- If REVIEW: Explain that their application is being reviewed by the hiring team. Mention what caught your attention. Set expectations for next steps.

Write ONLY the email body (no subject line, no "Dear" salutation — just the body text). Keep it to 3-5 sentences. Be specific to this candidate — do not write a generic template."""

    result = structured_llm_call(
        api_key=OPENROUTER_API_KEY,
        prompt=email_prompt,
        context_data={},
        output_schema={"email_body": "string — the email text"},
        model=MODEL,
        temperature=0.7,
    )
    if result['error']:
        return {"status": "error", "message": result['error']}
    
    email_body = result['result'].get('email_body', '')
    return {
        "status": "success",
        "email_body": email_body,
        "message": f"Email drafted ({len(email_body)} chars)",
        "usage": result['usage'],
    }


# Tool 3: Signal completion
def done(candidate_id):
    """Signal that processing is complete for this candidate."""
    return {"status": "success", "message": "Processing complete", "final": True}


# Build the tool registry
TOOL_REGISTRY = {
    "score_resume": {
        "function": score_resume,
        "description": "Score the candidate's resume against the job requirements. Returns a score (0-100) and reasoning. Call this FIRST.",
        "parameters": {
            "candidate_id": "string — the candidate's resume ID"
        }
    },
    "draft_outreach_email": {
        "function": draft_outreach_email,
        "description": "Draft a personalized outreach email based on the scoring outcome. Call this AFTER scoring.",
        "parameters": {
            "candidate_id": "string — the candidate's resume ID",
            "outcome": "string — must be INTERVIEW, REJECT, or REVIEW",
            "key_points": "string — key points about this candidate to mention in the email (from the score reasoning)"
        }
    },
    "done": {
        "function": done,
        "description": "Signal that all processing is complete for this candidate. Call this LAST, after scoring and drafting the email.",
        "parameters": {
            "candidate_id": "string — the candidate's resume ID"
        }
    },
}

print("Tools defined:")
for name, info in TOOL_REGISTRY.items():
    print(f"  - {name}: {info['description'][:60]}...")

## Step 2: The Agent Decision Function

The agent sees:
- The candidate ID
- What tools are available  
- What it has already done (action history)

It decides which tool to call next and with what parameters.

**Routing rules:**
- Score >= 80 → INTERVIEW
- Score 40-79 → REVIEW  
- Score < 40 → REJECT

In [ ]:
def agent_decide(api_key, candidate_id, action_history, tool_registry, model, temperature=0.3):
    """Agent decides which tool to call next."""
    
    tools_desc = "\n".join([
        f"- {name}: {info['description']}\n  Parameters: {json.dumps(info['parameters'])}"
        for name, info in tool_registry.items()
    ])
    
    history_str = "\n".join([
        f"Turn {i+1}: Called '{a['tool']}' → {a['result']['message']}"
        for i, a in enumerate(action_history)
    ]) if action_history else "No actions taken yet."
    
    prompt = f"""You are a hiring automation agent processing candidate {candidate_id}.

AVAILABLE TOOLS:
{tools_desc}

ACTION HISTORY:
{history_str}

ROUTING RULES:
- Score >= 80 → outcome is INTERVIEW
- Score 40-79 → outcome is REVIEW
- Score < 40 → outcome is REJECT

WORKFLOW:
1. First call score_resume to get the candidate's score
2. Then call draft_outreach_email with the appropriate outcome and key points from the scoring
3. Finally call done

Decide the NEXT action. You must follow the workflow order."""

    schema = {
        "tool": "string — name of tool to call",
        "parameters": "object — parameters for the tool",
        "reasoning": "string — why this action"
    }
    
    result = structured_llm_call(
        api_key=api_key,
        prompt=prompt,
        context_data={},
        output_schema=schema,
        model=model,
        temperature=temperature,
    )
    
    return result['result'], result.get('usage', {})

## Step 3: The Agent Loop

This is the core of any agent — a loop that:
1. Asks the LLM what to do
2. Executes the chosen tool
3. Logs the result
4. Repeats until done

In [ ]:
def run_agent(api_key, candidate_id, tool_registry, model, temperature=0.3, max_turns=6, verbose=True):
    """Run the agent loop for one candidate. Returns actions list and final results."""
    
    action_history = []
    total_cost = 0.0
    score = None
    outcome = None
    email_body = None
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"  Processing: {candidate_id}")
        print(f"{'='*70}")
    
    for turn in range(1, max_turns + 1):
        # Agent decides
        decision, usage = agent_decide(api_key, candidate_id, action_history, tool_registry, model, temperature)
        
        tool_name = decision.get('tool', '')
        params = decision.get('parameters', {})
        reasoning = decision.get('reasoning', '')
        
        if verbose:
            print(f"\n  Turn {turn}: {tool_name}")
            print(f"    Reasoning: {reasoning}")
        
        # Validate tool
        if tool_name not in tool_registry:
            if verbose:
                print(f"    ERROR: Unknown tool '{tool_name}'")
            break
        
        # Execute tool
        tool_fn = tool_registry[tool_name]['function']
        try:
            result = tool_fn(**params)
        except Exception as e:
            result = {"status": "error", "message": str(e)}
        
        if verbose:
            print(f"    Result: {result['message']}")
        
        # Track costs from tool usage (LLM-calling tools return usage)
        tool_usage = result.get('usage', {})
        
        # Extract key outputs
        if tool_name == 'score_resume' and result['status'] == 'success':
            score = result['score']
            outcome = 'INTERVIEW' if score >= 80 else ('REVIEW' if score >= 40 else 'REJECT')
            if verbose:
                print(f"    -> Score: {score}, Outcome: {outcome}")
        
        if tool_name == 'draft_outreach_email' and result['status'] == 'success':
            email_body = result['email_body']
        
        # Log action
        action_history.append({
            'turn': turn,
            'tool': tool_name,
            'parameters': params,
            'reasoning': reasoning,
            'result': result,
        })
        
        # Check if done
        if tool_name == 'done' or result.get('final'):
            break
    
    if verbose:
        print(f"\n  Done in {len(action_history)} turns")
        print(f"  Outcome: {outcome}")
        if email_body:
            print(f"\n  --- EMAIL ---")
            print(f"  {email_body[:200]}{'...' if len(email_body or '') > 200 else ''}")
            print(f"  --- END ---")
    
    return {
        'candidate_id': candidate_id,
        'score': score,
        'outcome': outcome,
        'email_body': email_body,
        'num_turns': len(action_history),
        'actions': action_history,
    }

## Step 4: Run the Agent

Let's process our candidates and watch the agent work.

In [ ]:
results = []

for cid in sorted(candidates.keys()):
    result = run_agent(
        api_key=OPENROUTER_API_KEY,
        candidate_id=cid,
        tool_registry=TOOL_REGISTRY,
        model=MODEL,
        temperature=0.3,
        verbose=True,
    )
    results.append(result)

print(f"\n\n{'='*70}")
print(f"Processed {len(results)} candidates")
print(f"{'='*70}")

## Step 5: Review Results

In [ ]:
# Summary table
summary = pd.DataFrame([{
    'candidate_id': r['candidate_id'],
    'tier': 'gold' if r['candidate_id'].startswith('g') else ('silver' if r['candidate_id'].startswith('s') else 'wild'),
    'score': r['score'],
    'outcome': r['outcome'],
    'turns': r['num_turns'],
    'email_preview': (r['email_body'] or '')[:80] + '...',
} for r in results])

print("Summary:")
display(summary)

print("\nOutcome distribution:")
print(summary.groupby(['tier', 'outcome']).size().unstack(fill_value=0))

## Step 6: Read the Emails

Let's look at the actual emails our agent drafted.

In [ ]:
for r in results:
    tier = 'GOLD' if r['candidate_id'].startswith('g') else ('SILVER' if r['candidate_id'].startswith('s') else 'WILD')
    color = {'GOLD': '#b8860b', 'SILVER': '#708090', 'WILD': '#333'}[tier]
    outcome_color = {'INTERVIEW': '#27ae60', 'REJECT': '#e74c3c', 'REVIEW': '#f39c12'}.get(r['outcome'], '#666')
    
    html = f"""
    <div style="border:1px solid #ddd; border-radius:8px; padding:16px; margin:12px 0; max-width:600px;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">
            <span style="font-weight:bold; color:{color};">[{tier}] {r['candidate_id']}</span>
            <span style="background:{outcome_color}; color:white; padding:2px 10px; border-radius:12px; font-size:0.85em;">{r['outcome']}</span>
        </div>
        <div style="font-size:0.85em; color:#666; margin-bottom:8px;">Score: {r['score']}/100</div>
        <div style="white-space:pre-wrap; line-height:1.5;">{r['email_body'] or 'No email generated'}</div>
    </div>
    """
    display(HTML(html))

## Step 7: Submit to Leaderboard

In [ ]:
for r in results:
    if r['outcome'] and r['email_body']:
        resp = submit_outreach(
            team_name=TEAM_NAME,
            resume_id=r['candidate_id'],
            outcome=r['outcome'],
            email_text=r['email_body'],
            score=r['score'],
            cost=None,  # TODO: track cost from usage
        )
        status = "OK" if 'error' not in resp else resp['error']
        print(f"  {r['candidate_id']}: {r['outcome']} -> {status}")

print(f"\nSubmitted {len(results)} results for team '{TEAM_NAME}'")
print("View results at: http://ai-leaderboard.site/lecture4")

## Step 8: Cleanup (if needed)

Run this cell to delete your team's submissions and re-submit.

In [ ]:
# Uncomment to delete your team's submissions:
# delete_team(TEAM_NAME)
# print(f"Deleted all submissions for team '{TEAM_NAME}'")

# Tasks

## Task 1: Read and Evaluate the Emails

Look at the emails your agent generated (Step 6 above).

**Questions:**
- Do the interview invites mention specific strengths from the candidate's resume?
- Are the rejections professional and respectful?
- Do gold candidates get interview invites? Do wild candidates get rejections?
- Which email would you most/least want to receive?

## Task 2: Improve the Email Prompt

The email drafting prompt is in the `draft_outreach_email` function (Step 1). Modify it to:
- Make interview invites mention the specific role (Senior Full-Stack Engineer)
- Make rejections more encouraging (suggest what skills to develop)
- Add a personal touch based on the candidate's background

Re-run the agent and compare the new emails to the originals. Submit your improved versions to the leaderboard.

## Task 3: Cost Optimization

The current agent uses Claude Sonnet for all three LLM calls (scoring decision, email drafting, agent decisions). That's expensive.

**Experiment:**
- Try using a cheaper model (e.g., `mistralai/mistral-small-2603`) for the email drafting tool while keeping Claude for scoring
- Try using a cheaper model for the agent decision function
- What's the cheapest configuration that still produces good emails?

**Hint:** You can set different models for different tools by modifying the `model` parameter in each tool function.

## Task 4: Run on More Candidates

Process all candidates and submit to the leaderboard. Compare your team's emails with other teams using the "View Emails" feature on the leaderboard.